# Qwen3.5-4B SQL QLoRA Fine-tuning — T4 GPU

Fine-tunes Qwen3.5-4B for NL→SQL using QLoRA (4-bit quantization + LoRA adapters).

**T4 specifics:**
- T4 compute capability 7.5: **fp16 only** — bfloat16 is NOT supported.
- 16 GB VRAM: batch_size=2 + grad_accum=8 → effective batch=16 at seq=1024.
- All checkpoints saved to Google Drive so Colab disconnects don't lose progress.

**Training correctness guarantee:**
- Completion-only loss via `DataCollatorForCompletionOnlyLM`.
- Correction chains have a WRONG SQL as context turn — it gets `label=-100` (masked).
- Only the final gold SQL is trained on.

**Estimated wall time:** ~3–4 hours (T4, seq=1024, 13k examples, 2 epochs).

---
Run cells in order. Do **not** skip Step 2 (pip install) or the Drive mount.

## Step 1 — Verify GPU

If `No devices were found`, go to **Runtime → Change runtime type → T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    print(f'bf16 supported: {torch.cuda.is_bf16_supported()}')
else:
    raise RuntimeError('No GPU found. Change runtime to T4 GPU before proceeding.')

## Step 2 — Install dependencies

Torch is preinstalled on Colab. Installing packages (~3 min).
**Restart runtime after installation completes (Runtime → Restart session).**

> **Note:** Qwen3.5 requires transformers>=5.2.0. Earlier versions fail with
> `ValueError: model type 'qwen3_5' not recognized`.

In [ ]:
# Restart runtime after this cell completes
!pip install -q \n    "transformers>=5.2.0" \n    "peft>=0.19.0" \n    "trl>=1.0.0" \n    "bitsandbytes>=0.49.0" \n    "accelerate>=1.4.0" \n    "datasets>=4.7.0" \n    "tokenizers>=0.21.0" \n    "sentencepiece>=0.2.0" \n    "safetensors>=0.4.5"

import importlib
for pkg in ["transformers","peft","trl","bitsandbytes","accelerate"]:
    v = importlib.import_module(pkg).__version__
    print(f"{pkg}: {v}")

## Step 3 — Mount Google Drive

All data, checkpoints, and adapters stored on Drive so VM restarts can resume.
You will be prompted to authorize Drive access.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import pathlib
drive_root = pathlib.Path('/content/drive/MyDrive')
print(f'Drive mounted: {drive_root.exists()}')

## Step 4 — Configure paths

Edit `DATA_DIR` and `OUTPUT_DIR` to match your Drive layout.

**Upload to Drive before Step 7:**
```
MyDrive/sql-agent-data/
    train.prompt.jsonl   (~25 MB)
    val.prompt.jsonl     (~1.7 MB)
    test.prompt.jsonl    (~1.9 MB)
```
Generate locally: `python colab/prepare_data.py --max-seq 1024`

**Also upload scripts to Drive:**
```
MyDrive/sql-agent-scripts/
    train_qlora.py
    merge_adapter.py
    eval_exec.py
```

In [ ]:
import pathlib

# ---- EDIT THESE ----
BASE_MODEL  = 'Qwen/Qwen3.5-4B'   # HF hub id (downloads ~8 GB) or Drive path
DATA_DIR    = '/content/drive/MyDrive/sql-agent-data'
OUTPUT_DIR  = '/content/drive/MyDrive/sql-agent-adapters/qwen-sql-qlora'
SCRIPTS_DIR = '/content/drive/MyDrive/sql-agent-scripts'
MAX_SEQ     = 1024    # increase to 1536/2048 if VRAM allows
NUM_EPOCHS  = 2
# ---------------------

DATA_DIR   = pathlib.Path(DATA_DIR)
OUTPUT_DIR = pathlib.Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

TRAIN_FILE = DATA_DIR / 'train.prompt.jsonl'
VAL_FILE   = DATA_DIR / 'val.prompt.jsonl'
TEST_FILE  = DATA_DIR / 'test.prompt.jsonl'

for f in [TRAIN_FILE, VAL_FILE, TEST_FILE]:
    status = 'OK' if f.exists() else 'MISSING — upload to Drive first'
    print(f'{f.name}: {status}')
print(f'\nOutput dir: {OUTPUT_DIR}')
print(f'Base model: {BASE_MODEL}')
print(f'Max seq:    {MAX_SEQ}')

## Step 5 — Verify data format

Inspect prompt/completion format and count examples. Training starts with Step 7.

In [ ]:
import json

def inspect_jsonl(path, n=2):
    with open(path) as f:
        for i, line in enumerate(f):
            if i >= n: break
            ex = json.loads(line)
            print(f'[{i}] prompt tail:  {repr(ex["prompt"][-80:])}')
            print(f'     completion:   {repr(ex["completion"][:60])}')

print('=== Train sample ===')
inspect_jsonl(TRAIN_FILE)

for name, path in [('train', TRAIN_FILE), ('val', VAL_FILE), ('test', TEST_FILE)]:
    count = sum(1 for _ in open(path))
    print(f'{name}: {count} examples')

## Step 6 — (Optional) Run prepare_data on Colab

Skip this if you already uploaded pre-built `.prompt.jsonl` files.
Only needed if you prefer to render data on Colab itself.

In [ ]:
# OPTIONAL — skip if prompt JSONL already on Drive
RAW_DATA_DIR = '/content/drive/MyDrive/sql-agent-raw'  # adjust path

!python {SCRIPTS_DIR}/prepare_data.py \
    --data-dir  {RAW_DATA_DIR} \
    --out-dir   {DATA_DIR} \
    --tokenizer {BASE_MODEL} \
    --max-seq   {MAX_SEQ}

## Step 7 — Train QLoRA

Main training cell. Saves checkpoints every 200 steps to Drive.

**T4 training config:**
- `fp16=True, bf16=False` — T4 (CC 7.5) lacks bf16
- Effective batch = `per_device(2) × grad_accum(8)` = 16
- ~3–4 hours for 2 epochs on full dataset

**After Colab disconnect:** reconnect, re-run Steps 1–4, set `RESUME = True` below.

In [ ]:
RESUME = False  # Set True to resume from latest checkpoint after a disconnect

resume_flag = '--resume-from-checkpoint latest' if RESUME else ''

!python {SCRIPTS_DIR}/train_qlora.py \
    --base-model                  {BASE_MODEL} \
    --train-file                  {TRAIN_FILE} \
    --val-file                    {VAL_FILE} \
    --output-dir                  {OUTPUT_DIR} \
    --max-seq-length              {MAX_SEQ} \
    --num-epochs                  {NUM_EPOCHS} \
    --per-device-batch-size       2 \
    --gradient-accumulation-steps 8 \
    --learning-rate               2e-4 \
    --warmup-ratio                0.03 \
    --lr-scheduler                cosine \
    --logging-steps               10 \
    --eval-steps                  100 \
    --save-steps                  200 \
    {resume_flag}

## Step 8 — Plot training loss

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt

def load_trainer_logs(output_dir):
    state_files = sorted(pathlib.Path(output_dir).rglob('trainer_state.json'))
    if not state_files:
        return [], []
    with open(state_files[-1]) as f:
        state = json.load(f)
    train_loss = [(e['step'],e['loss']) for e in state.get('log_history',[]) if 'loss' in e and 'eval_loss' not in e]
    eval_loss  = [(e['step'],e['eval_loss']) for e in state.get('log_history',[]) if 'eval_loss' in e]
    return train_loss, eval_loss

train_loss, eval_loss = load_trainer_logs(OUTPUT_DIR)
if train_loss:
    fig, ax = plt.subplots(figsize=(10,5))
    steps_t, vals_t = zip(*train_loss)
    ax.plot(steps_t, vals_t, label='Train loss', alpha=0.7, linewidth=1)
    if eval_loss:
        steps_e, vals_e = zip(*eval_loss)
        ax.plot(steps_e, vals_e, label='Eval loss', linewidth=2, color='red')
    ax.set_xlabel('Steps'); ax.set_ylabel('Loss')
    ax.set_title('Qwen3.5-4B QLoRA — sql-agent')
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(str(OUTPUT_DIR / 'loss_curve.png'), dpi=150)
    plt.show()
    print(f'Final train loss: {vals_t[-1]:.4f}')
    if eval_loss: print(f'Final eval loss:  {vals_e[-1]:.4f}')
else:
    print('No loss data found. Run training first.')

## Step 9 — Merge adapter (optional)

Merges LoRA into base model. Requires ~16 GB RAM (run on High-RAM runtime).
**Skip if using the adapter directly** via `PeftModel.from_pretrained()`.

In [ ]:
MERGED_DIR  = '/content/drive/MyDrive/sql-agent-models/qwen-sql-merged'
ADAPTER_DIR = str(OUTPUT_DIR / 'final_adapter')

!python {SCRIPTS_DIR}/merge_adapter.py \
    --base-model  {BASE_MODEL} \
    --adapter-dir {ADAPTER_DIR} \
    --output-dir  {MERGED_DIR} \
    --device      cpu

import pathlib
for f in sorted(pathlib.Path(MERGED_DIR).iterdir()):
    print(f'{f.name}: {f.stat().st_size/1e6:.1f} MB')

## Step 10 — Optional: Execution-accuracy evaluation

Requires Spider/BIRD SQLite DB files on Drive:
`MyDrive/spider_data/database/<db_id>/<db_id>.sqlite`

~30–60 min for 1,029 test examples on T4.

In [ ]:
DB_ROOT    = '/content/drive/MyDrive/spider_data/database'
TEST_SRC   = '/content/drive/MyDrive/sql-agent-raw/test.jsonl'
EVAL_OUT   = str(OUTPUT_DIR / 'eval_results.jsonl')
ADAPTER_DIR = str(OUTPUT_DIR / 'final_adapter')

!python {SCRIPTS_DIR}/eval_exec.py \
    --test-file   {TEST_FILE} \
    --test-src    {TEST_SRC} \
    --db-root     {DB_ROOT} \
    --base-model  {BASE_MODEL} \
    --adapter-dir {ADAPTER_DIR} \
    --output-file {EVAL_OUT}

import json
results = [json.loads(l) for l in open(EVAL_OUT)]
n = len(results)
strict   = sum(1 for r in results if r['exec_strict'])
superset = sum(1 for r in results if r['exec_superset'])
errors   = sum(1 for r in results if r['pred_error'])
print(f'Results on {n} examples:')
print(f'  Exec acc (strict):   {strict}/{n} = {100*strict/n:.1f}%')
print(f'  Exec acc (superset): {superset}/{n} = {100*superset/n:.1f}%')
print(f'  Prediction errors:   {errors}/{n} = {100*errors/n:.1f}%')

## Next Steps

1. Download `final_adapter/` from Drive → `sql-agent/adapters/qwen-sql-qlora/final_adapter/`
2. In `src/qwen_model_fn.py`: `make_qwen_model_fn(adapter_path='adapters/qwen-sql-qlora/final_adapter')`
3. Reconcile inference format: update `qwen_model_fn.py` to use `enable_thinking=False` in `apply_chat_template` instead of the manual `<think></think>` prefill — this exactly matches training distribution.
4. Run `src/eval_harness.py` on M1 to compare base vs fine-tuned execution accuracy.

See `colab/README.md` for full details.